In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

In [ ]:
project_id = 'oceanic-citadel-454608-d2'
from google.cloud import bigquery
client = bigquery.Client(project=project_id)

In [ ]:
import pandas as pd
import numpy as np

# ── Date anchor ──
AS_OF_DATE = pd.Timestamp.now().normalize() - pd.Timedelta(days=2)

# ── Populations for ARPU curves (PPC excluded; Organic excluded — handled via organic share) ──
POPULATIONS   = ['Web', 'App', 'Affiliate']
ORGANIC_LABEL = 'Organic'

# ── Patch windows (s, e) ──
PATCHES = (
    (1, 7), (7, 14), (14, 30), (30, 60), (60, 90),
    (90, 120), (120, 150), (150, 180), (180, 270), (270, 365),
)

# ── Goal horizons & checkpoints ──
GOAL_HORIZONS = [7, 30, 60, 90, 120, 150, 180, 210, 240, 270, 365]
CHECKPOINTS   = [7, 30, 60, 90, 120, 150, 180, 210, 240, 270, 365]

# ── Trim config per population ──
TRIM_CONFIG = {
    'App':       {'method': 'winsor', 'pct': 0},
    'Web':       {'method': 'winsor', 'pct': 0.01},
    'Affiliate': {'method': 'winsor', 'pct': 0.01},
    'Blended':   {'method': 'winsor', 'pct': 0},
}

# ── Organic share trim ──
ORGANIC_TRIM_METHOD = 'winsor'   # 'winsor' or 'cohort_trim'
ORGANIC_TRIM_PCT    = 0

# ── Organic share horizon cap ──
# App attribution changed ~2025-08-12; cohorts before that have inflated organic %.
# For horizons above this cap, pin organic share to the cap horizon's value.
ORGANIC_SHARE_CAP_HORIZON = 120

# ── CV thresholds ──
CV_THRESHOLD        = 0.15
CV_GOOD_ENOUGH      = 0.10
MAX_REMOVE_FRACTION = 0.15

# ── Lookback window ──
LOOKBACK_COHORTS = 35

print(f'Config loaded. as_of_date = {AS_OF_DATE.date()}')
print('Mode: PERSISTENT TRIM (excluded users carry forward, no fallback windows)')

In [ ]:
from pandas_gbq import read_gbq

# Earliest date needed = AS_OF_DATE − (longest_horizon + lookback − 1), with a 5-day buffer.
SQL_FLOOR_DATE = (
    AS_OF_DATE - pd.Timedelta(days=max(GOAL_HORIZONS) + LOOKBACK_COHORTS + 5)
).date()
print(f'SQL floor date: {SQL_FLOOR_DATE}  (AS_OF_DATE = {AS_OF_DATE.date()})')

# `analytics.realprize_cost_per_user` is pre-filtered for test_account = 0 AND
# marketing_account = 0 — no need to join the users table or apply those filters.
# TikTok (affid 4313) is excluded entirely.
# PPC is included so its revenue counts in the organic-share denominator,
# but it's excluded from POPULATIONS for ARPU curve building.
users_df = read_gbq(f"""
SELECT
  id,
  CASE
    WHEN affid IN (63, 2521, 2535, 4957, 4971, 5048, 5062, 5069) THEN 'Web'
    WHEN affid = 1                                    THEN 'App'
    WHEN affid IN (64, 71)                            THEN 'PPC'
    WHEN affid IN (0, 78, 2290)                       THEN 'Organic'
    ELSE 'Affiliate'
  END AS population,
  CASE WHEN affid = 1 THEN 'app' ELSE 'non_app' END AS scope,
  CASE
    WHEN affid = 1 AND channel_type = 'app_organic' THEN 'organic'
    WHEN affid = 1                                   THEN 'acquired'
    WHEN affid IN (0, 78, 2290)                      THEN 'organic'
    ELSE 'acquired'
  END AS bucket,
  DATE(MIN(cost_date)) AS cost_date
FROM `analytics.realprize_cost_per_user`
WHERE cost_date >= DATE('{SQL_FLOOR_DATE}')
  AND affid != 4313
  AND id > 0
GROUP BY id, population, scope, bucket
""", project_id=project_id, use_bqstorage_api=True)

revenue_df = read_gbq(f"""
SELECT
  playerId AS playerid,
  DATE(date) AS date,
  SUM(amount) / 100.0 AS amount
FROM `realprize.casino_astropay_dmn`
WHERE Status = 'APPROVED'
  AND date >= DATE('{SQL_FLOOR_DATE}')
GROUP BY 1, 2
""", project_id=project_id, use_bqstorage_api=True)

print(f'users_df:   {len(users_df):,} rows')
print(f'revenue_df: {len(revenue_df):,} rows')
print(users_df['population'].value_counts().to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════
# HELPERS — math & base table construction
# ══════════════════════════════════════════════════════════════

def weighted_mean_std_cv(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    m = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x, w = x[m], w[m]
    if x.size == 0:
        return np.nan, np.nan, np.nan
    mu  = np.average(x, weights=w)
    var = np.average((x - mu) ** 2, weights=w)
    sd  = np.sqrt(var)
    cv  = sd / mu if mu != 0 else np.nan
    return mu, sd, cv


def build_user_revenue_cums(users_df, revenue_df, *, max_day=365):
    """
    Precompute cumulative per-user revenue indexed by (population, cost_date, user, dsi).
    Returns:
      u          — one row per (population, user) with their earliest cost_date
      daily_user — cumulative revenue at each (population, cost_date, user, dsi)
    """
    u = users_df[['id', 'population', 'cost_date']].copy()
    u['population'] = u['population'].astype(str).str.strip()
    u['cost_date']  = pd.to_datetime(u['cost_date'], errors='coerce').dt.date
    u = u.loc[pd.notna(u['cost_date'])].copy()
    u = u.groupby(['population', 'id'], as_index=False)['cost_date'].min()
    u = u.rename(columns={'id': '__uid__'})

    r = revenue_df[['playerid', 'date', 'amount']].copy()
    r['date'] = pd.to_datetime(r['date'], errors='coerce').dt.date
    r = r.loc[pd.notna(r['date'])].copy()

    rr = r.merge(u, left_on='playerid', right_on='__uid__', how='inner')
    rr['dsi'] = (pd.to_datetime(rr['date']) - pd.to_datetime(rr['cost_date'])).dt.days
    rr = rr.loc[(rr['dsi'] >= 0) & (rr['dsi'] <= (max_day - 1))].copy()

    daily_user = (
        rr.groupby(['population', 'cost_date', '__uid__', 'dsi'], observed=True)['amount']
          .sum().reset_index()
          .sort_values(['population', 'cost_date', '__uid__', 'dsi'])
    )
    daily_user['cum_amount'] = (
        daily_user.groupby(['population', 'cost_date', '__uid__'], observed=True)['amount']
                  .cumsum()
    )
    return u, daily_user


print('Math + base table helpers defined.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# HELPERS — trimming & cohort revenue summation
# ══════════════════════════════════════════════════════════════

def compute_winsor_caps(daily_user_cums, cohort_users, e, top_pct=0.01):
    """Cap per-user cumulative revenue at the (1-top_pct) quantile within each cohort date."""
    if top_pct <= 0:
        caps = cohort_users[['population', 'cost_date', '__uid__']].copy()
        caps['cap_e'] = np.inf
        return caps
    du = daily_user_cums.loc[daily_user_cums['dsi'] <= (e - 1)].copy()
    if du.empty:
        caps = cohort_users[['population', 'cost_date', '__uid__']].copy()
        caps['cap_e'] = np.inf
        return caps
    per_user = (
        du.groupby(['population', 'cost_date', '__uid__'], observed=True)['cum_amount']
          .max().reset_index(name='cum_e')
    )
    per_user = cohort_users.merge(per_user, on=['population', 'cost_date', '__uid__'], how='left')
    per_user['cum_e'] = per_user['cum_e'].fillna(0.0)
    per_user['cap_e'] = (
        per_user.groupby(['population', 'cost_date'], observed=True)['cum_e']
                .transform(lambda s: s[s > 0].quantile(1.0 - top_pct) if (s > 0).any() else np.inf)
    )
    return per_user[['population', 'cost_date', '__uid__', 'cap_e']]


def apply_cohort_trim(daily_user_cums, cohort_users, e, trim_pct=0.10):
    """Remove the top trim_pct of depositors (by cumulative revenue) from each cohort date."""
    du = daily_user_cums.loc[daily_user_cums['dsi'] <= (e - 1)].copy()
    if du.empty:
        return cohort_users.copy()
    per_user = (
        du.groupby(['population', 'cost_date', '__uid__'], observed=True)['cum_amount']
          .max().reset_index(name='cum_e')
    )
    per_user = cohort_users.merge(per_user, on=['population', 'cost_date', '__uid__'], how='left')
    per_user['cum_e'] = per_user['cum_e'].fillna(0.0)
    depositors = per_user.loc[per_user['cum_e'] > 0].copy()
    if depositors.empty:
        return cohort_users.copy()
    thresholds = (
        depositors.groupby(['population', 'cost_date'], observed=True)['cum_e']
                  .quantile(1.0 - trim_pct).reset_index(name='threshold')
    )
    per_user = per_user.merge(thresholds, on=['population', 'cost_date'], how='left')
    per_user['threshold'] = per_user['threshold'].fillna(np.inf)
    keep = per_user.loc[
        (per_user['cum_e'] == 0) | (per_user['cum_e'] <= per_user['threshold'])
    ][['population', 'cost_date', '__uid__']]
    return keep.copy()


def get_trimmed_cohort_and_caps(population, cohort_users, daily_user_cums, e):
    cfg = TRIM_CONFIG.get(population, {'method': 'cohort_trim', 'pct': 0.10})
    caps, trimmed = None, cohort_users
    if cfg['method'] == 'winsor':
        caps = compute_winsor_caps(daily_user_cums, cohort_users, e, top_pct=cfg['pct'])
    elif cfg['method'] == 'cohort_trim':
        trimmed = apply_cohort_trim(daily_user_cums, cohort_users, e, trim_pct=cfg['pct'])
    return trimmed, caps


def sum_cum_at_idx(daily_user_cums, *, cohort_users, idx, caps=None):
    """Sum cumulative revenue across all users in cohort_users at day index idx."""
    grp_cols = ['population', 'cost_date']
    if idx < 0:
        out = cohort_users.groupby(grp_cols, observed=True)['__uid__'].nunique().reset_index()
        out['sum_cum'] = 0.0
        return out[grp_cols + ['sum_cum']]
    du = daily_user_cums.loc[daily_user_cums['dsi'] <= idx].copy()
    if du.empty:
        out = cohort_users.groupby(grp_cols, observed=True)['__uid__'].nunique().reset_index()
        out['sum_cum'] = 0.0
        return out[grp_cols + ['sum_cum']]
    per_user = (
        du.groupby(grp_cols + ['__uid__'], observed=True)['cum_amount']
          .max().reset_index(name='cum')
    )
    per_user = cohort_users.merge(per_user, on=grp_cols + ['__uid__'], how='left')
    per_user['cum'] = per_user['cum'].fillna(0.0)
    if caps is not None:
        per_user = per_user.merge(caps, on=grp_cols + ['__uid__'], how='left')
        per_user['cap_e'] = per_user['cap_e'].fillna(np.inf)
        per_user['cum']   = np.minimum(per_user['cum'], per_user['cap_e'])
    sums = per_user.groupby(grp_cols, observed=True)['cum'].sum().reset_index(name='sum_cum')
    return sums


print('Trim + summation helpers defined.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# ADAPTIVE CV ANALYSIS — persistent trim
# ══════════════════════════════════════════════════════════════

def patch_cv_adaptive(
    u_base, daily_user_cums, *,
    population, s, e, as_of_date,
    excluded_uids=None,
    lookback_cohorts=LOOKBACK_COHORTS,
    cv_threshold=CV_THRESHOLD,
    cv_good_enough=CV_GOOD_ENOUGH,
    max_remove_fraction=MAX_REMOVE_FRACTION,
    debug=True,
):
    as_of_date   = pd.to_datetime(as_of_date).normalize()
    cohort_end   = (as_of_date - pd.Timedelta(days=e)).date()
    cohort_start = (as_of_date - pd.Timedelta(days=e + (lookback_cohorts - 1))).date()

    cohort_users = u_base.loc[
        (u_base['population'] == population) &
        (u_base['cost_date'] >= cohort_start) &
        (u_base['cost_date'] <= cohort_end)
    ][['population', 'cost_date', '__uid__']].copy()

    if cohort_users.empty:
        return pd.DataFrame(), {}, [], False, set()

    all_cohort_users = cohort_users.copy()
    n_users_in_cohort = int(cohort_users['__uid__'].nunique())

    if excluded_uids:
        cohort_users = cohort_users.loc[
            ~cohort_users['__uid__'].isin(excluded_uids)
        ].copy()

    n_users_after_prior = int(cohort_users['__uid__'].nunique())
    n_users_excluded_prior = n_users_in_cohort - n_users_after_prior

    if cohort_users.empty:
        return pd.DataFrame(), {}, [], False, set()

    trimmed_users, caps = get_trimmed_cohort_and_caps(
        population, cohort_users, daily_user_cums, e
    )
    n_users_pre_trim  = n_users_after_prior
    n_users_post_trim = int(trimmed_users['__uid__'].nunique())

    newly_excluded = (
        set(cohort_users['__uid__'].unique()) - set(trimmed_users['__uid__'].unique())
    )

    denom_w = (
        trimmed_users.groupby(['population', 'cost_date'], observed=True)['__uid__']
                     .nunique().reset_index(name='N_users')
    )
    sum_s = sum_cum_at_idx(
        daily_user_cums, cohort_users=trimmed_users, idx=s - 1, caps=caps
    ).rename(columns={'sum_cum': 'sum_cum_s'})
    sum_e = sum_cum_at_idx(
        daily_user_cums, cohort_users=trimmed_users, idx=e - 1, caps=caps
    ).rename(columns={'sum_cum': 'sum_cum_e'})

    sum_e_all = sum_cum_at_idx(
        daily_user_cums, cohort_users=all_cohort_users, idx=e - 1, caps=None
    ).rename(columns={'sum_cum': 'sum_cum_e_all'})
    total_rev_before_trim = float(sum_e_all['sum_cum_e_all'].sum())

    patch = (
        denom_w
        .merge(sum_s, on=['population', 'cost_date'])
        .merge(sum_e, on=['population', 'cost_date'])
    )
    patch['ARPU_s']       = patch['sum_cum_s'] / patch['N_users']
    patch['ARPU_e']       = patch['sum_cum_e'] / patch['N_users']
    patch['growth_ratio'] = np.where(
        patch['ARPU_s'] > 0, patch['ARPU_e'] / patch['ARPU_s'], np.nan
    )

    _, _, cv_before = weighted_mean_std_cv(patch['growth_ratio'].values, patch['sum_cum_s'].values)

    mu_unw = np.nanmean(patch['growth_ratio'].values)
    patch['abs_dev'] = (patch['growth_ratio'] - mu_unw).abs()
    sorted_dates  = patch.sort_values('abs_dev', ascending=False)['cost_date'].tolist()
    max_removable = max(1, int(np.floor(len(patch) * max_remove_fraction)))

    removed   = []
    remaining = patch.copy()

    for candidate in sorted_dates:
        _, _, cv_now = weighted_mean_std_cv(
            remaining['growth_ratio'].values, remaining['sum_cum_s'].values
        )
        if np.isnan(cv_now) or cv_now <= cv_good_enough:
            break
        if len(removed) >= max_removable:
            break
        removed.append(candidate)
        remaining = remaining.loc[~remaining['cost_date'].isin(removed)]

    mean_a, _, cv_after = weighted_mean_std_cv(
        remaining['growth_ratio'].values, remaining['sum_cum_s'].values
    )
    flagged = (not np.isnan(cv_after)) and (cv_after > cv_threshold)
    total_rev_after_trim = float(patch['sum_cum_e'].sum())
    cfg = TRIM_CONFIG.get(population, {})

    if debug:
        flag_tag = f'  >>> FLAGGED (cv={cv_after:.4f} > {cv_threshold})' if flagged else ''
        print(
            f'  [{population}] {s}->{e}  '
            f'cv {cv_before:.4f}->{cv_after:.4f}  '
            f'removed={len(removed)}/{len(patch)}  '
            f'excl_prior={n_users_excluded_prior:,}  '
            f'pre/post={n_users_pre_trim:,}/{n_users_post_trim:,}  '
            f'newly_excl={len(newly_excluded):,}{flag_tag}'
        )

    stats = dict(
        population              = population,
        patch                   = f'{s}->{e}',
        cohort_start            = str(cohort_start),
        cohort_end              = str(cohort_end),
        n_cohort_dates_total    = int(len(patch)),
        n_cohort_dates_kept     = int(len(remaining)),
        n_users_excluded_prior  = n_users_excluded_prior,
        n_users_pre_trim        = n_users_pre_trim,
        n_users_post_trim       = n_users_post_trim,
        n_users_dropped_by_trim = n_users_pre_trim - n_users_post_trim,
        total_rev_before_trim   = total_rev_before_trim,
        total_rev_after_trim    = total_rev_after_trim,
        cv_before               = float(cv_before) if not np.isnan(cv_before) else None,
        cv_after                = float(cv_after)  if not np.isnan(cv_after)  else None,
        mean_after              = float(mean_a)    if not np.isnan(mean_a)    else None,
        flagged                 = bool(flagged),
        removed_dates           = removed,
        trim_method             = cfg.get('method', 'none'),
        trim_pct                = cfg.get('pct', 0),
    )
    return patch, stats, removed, flagged, newly_excluded


print('Adaptive CV function defined (persistent trim).')

In [ ]:
# ══════════════════════════════════════════════════════════════
# PERSISTENT TRIM CURVE BUILDER
# ══════════════════════════════════════════════════════════════

def build_curve(
    u_base, daily_user_cums, *,
    population, as_of_date,
    lookback_cohorts=LOOKBACK_COHORTS,
    cv_threshold=CV_THRESHOLD, cv_good_enough=CV_GOOD_ENOUGH,
    max_remove_fraction=MAX_REMOVE_FRACTION, debug=True,
):
    as_of_date = pd.to_datetime(as_of_date).normalize()
    cv_rows      = []
    effective    = []
    excluded_uids = set()

    for (s, e) in PATCHES:
        patch, stats, removed, flagged, newly_excluded = patch_cv_adaptive(
            u_base, daily_user_cums,
            population=population, s=s, e=e,
            as_of_date=as_of_date,
            excluded_uids=excluded_uids,
            lookback_cohorts=lookback_cohorts,
            cv_threshold=cv_threshold, cv_good_enough=cv_good_enough,
            max_remove_fraction=max_remove_fraction, debug=debug,
        )
        excluded_uids |= newly_excluded
        if not stats:
            continue
        min_cohort_dates = globals().get('MIN_COHORT_DATES', 1)
        if stats.get('n_cohort_dates_total', 0) < min_cohort_dates:
            if debug:
                print(f'  [{population}] {s}->{e}  insufficient data — skipping')
            continue
        cv_rows.append(stats)
        effective.append({
            's': s, 'e': e,
            'removed_dates': removed,
            'excluded_snapshot': frozenset(excluded_uids),
        })

    if not effective:
        return pd.DataFrame(cv_rows), pd.DataFrame()
    if debug:
        print(f'  Total unique users excluded across all patches: {len(excluded_uids):,}')

    step_rows = []
    for ep in effective:
        s, e      = ep['s'], ep['e']
        bad_dates = set(ep['removed_dates'])
        excl      = ep['excluded_snapshot']
        start_k   = 2 if s == 1 else (s + 1)
        cohort_end   = (as_of_date - pd.Timedelta(days=e)).date()
        cohort_start = (as_of_date - pd.Timedelta(days=e + (lookback_cohorts - 1))).date()
        cohort_users = u_base.loc[
            (u_base['population'] == population) &
            (u_base['cost_date'] >= cohort_start) &
            (u_base['cost_date'] <= cohort_end)
        ][['population', 'cost_date', '__uid__']].copy()
        if bad_dates:
            cohort_users = cohort_users.loc[~cohort_users['cost_date'].isin(bad_dates)].copy()
        if excl:
            cohort_users = cohort_users.loc[~cohort_users['__uid__'].isin(excl)].copy()
        if cohort_users.empty:
            continue
        _, caps = get_trimmed_cohort_and_caps(population, cohort_users, daily_user_cums, e)
        denom_w = (
            cohort_users.groupby(['population', 'cost_date'], observed=True)['__uid__']
                        .nunique().reset_index(name='N_users')
        )
        for k in range(start_k, e + 1):
            sum_prev = sum_cum_at_idx(
                daily_user_cums, cohort_users=cohort_users, idx=k - 2, caps=caps
            ).rename(columns={'sum_cum': 'sum_prev'})
            sum_curr = sum_cum_at_idx(
                daily_user_cums, cohort_users=cohort_users, idx=k - 1, caps=caps
            ).rename(columns={'sum_cum': 'sum_curr'})
            tmp = (
                denom_w
                .merge(sum_prev, on=['population', 'cost_date'])
                .merge(sum_curr, on=['population', 'cost_date'])
            )
            tmp['ARPU_prev']  = tmp['sum_prev'] / tmp['N_users']
            tmp['ARPU_curr']  = tmp['sum_curr'] / tmp['N_users']
            tmp['step_ratio'] = np.where(
                tmp['ARPU_prev'] > 0, tmp['ARPU_curr'] / tmp['ARPU_prev'], np.nan
            )
            mean_step, _, _ = weighted_mean_std_cv(
                tmp['step_ratio'].values, tmp['sum_prev'].values
            )
            step_rows.append({
                'population': population, 'day': int(k),
                'growth_step': float(mean_step), 'effective_patch': f'{s}->{e}',
            })

    if not step_rows:
        return pd.DataFrame(cv_rows), pd.DataFrame()

    step_df = pd.DataFrame(step_rows).sort_values('day').reset_index(drop=True)
    first      = effective[0]
    fs, fe     = first['s'], first['e']
    excl_first = first['excluded_snapshot']
    base_end   = (as_of_date - pd.Timedelta(days=fe)).date()
    base_start = (as_of_date - pd.Timedelta(days=fe + (lookback_cohorts - 1))).date()
    base_users = u_base.loc[
        (u_base['population'] == population) &
        (u_base['cost_date'] >= base_start) &
        (u_base['cost_date'] <= base_end)
    ][['population', 'cost_date', '__uid__']].copy()
    bad_first = set(first['removed_dates'])
    if bad_first:
        base_users = base_users.loc[~base_users['cost_date'].isin(bad_first)].copy()
    if excl_first:
        base_users = base_users.loc[~base_users['__uid__'].isin(excl_first)].copy()
    _, base_caps = get_trimmed_cohort_and_caps(population, base_users, daily_user_cums, fe)
    denom_base = (
        base_users.groupby(['population', 'cost_date'], observed=True)['__uid__']
                  .nunique().reset_index(name='N_users')
    )
    start_idx = 0 if fs == 1 else (fs - 1)
    sum_day1 = sum_cum_at_idx(
        daily_user_cums, cohort_users=base_users, idx=start_idx, caps=base_caps
    ).rename(columns={'sum_cum': 'sum_day1'})
    base = denom_base.merge(sum_day1, on=['population', 'cost_date'], how='inner')
    pooled_arpu_1 = (
        base['sum_day1'].sum() / base['N_users'].sum()
        if base['N_users'].sum() > 0 else 0.0
    )
    start_day = 1 if fs == 1 else fs
    out_rows  = [{'population': population, 'day': start_day,
                  'ARPU_nominal': float(pooled_arpu_1),
                  'growth_step':  np.nan,
                  'effective_patch': f'{fs}->{fe}'}]
    arpu = float(pooled_arpu_1)
    for _, row in step_df.iterrows():
        g = row['growth_step']
        if not np.isfinite(g):
            continue
        arpu *= float(g)
        out_rows.append({'population': population, 'day': int(row['day']),
                         'ARPU_nominal': float(arpu),
                         'growth_step':  float(g),
                         'effective_patch': row['effective_patch']})
    curve = pd.DataFrame(out_rows).sort_values('day').reset_index(drop=True)
    all_days = pd.DataFrame({'day': range(start_day, 366)})
    curve    = all_days.merge(curve, on='day', how='left')
    curve['population']      = population
    curve['effective_patch'] = curve['effective_patch'].ffill()
    curve['ARPU_nominal']    = curve['ARPU_nominal'].interpolate(
        method='linear', limit_area='inside'
    )
    curve = curve.dropna(subset=['ARPU_nominal']).reset_index(drop=True)
    curve['is_extrapolated'] = False
    return pd.DataFrame(cv_rows), curve


def build_all_populations(u_base, daily_user_cums, *, as_of_date, debug=True):
    all_cv_rows = []
    all_curves  = []
    for pop in POPULATIONS:
        if debug:
            print(f'\n{"=" * 55}')
            print(f'POPULATION: {pop}')
            print(f'{"=" * 55}')
        cv_df, curve = build_curve(
            u_base, daily_user_cums,
            population=pop, as_of_date=as_of_date, debug=debug,
        )
        if not cv_df.empty:
            all_cv_rows.append(cv_df)
        if not curve.empty:
            all_curves.append(curve)
    cv_out    = pd.concat(all_cv_rows, ignore_index=True) if all_cv_rows else pd.DataFrame()
    curve_out = pd.concat(all_curves, ignore_index=True) if all_curves else pd.DataFrame()
    return cv_out, curve_out

print('Persistent trim curve builder defined.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# ORGANIC SHARE — multi-config cohort progression with scope
# Supports winsor, cohort_trim (with persistent trim), or none.
# Percentiles always computed from depositors only.
#
# If users_df has 'scope' and 'bucket' columns, uses them directly
# (RP: app vs non_app). Otherwise defaults to scope='all' and
# derives bucket from population == ORGANIC_LABEL (LS).
# ══════════════════════════════════════════════════════════════

def organic_share_cohort_progression(
    users_df, revenue_df, *,
    as_of_date,
    goal_horizons=GOAL_HORIZONS,
    checkpoints=CHECKPOINTS,
    lookback_cohorts=LOOKBACK_COHORTS,
    positive_amount_only=True,
    trim_configs,
    persistent_trim=True,
):
    as_of_date = pd.to_datetime(as_of_date).normalize()

    has_scope = 'scope' in users_df.columns and 'bucket' in users_df.columns
    cols = ['id', 'cost_date']
    if has_scope:
        cols += ['scope', 'bucket']
    if 'population' in users_df.columns:
        cols.append('population')

    u = users_df[cols].copy()
    u['cost_date'] = pd.to_datetime(u['cost_date'], errors='coerce').dt.date
    u = u.loc[pd.notna(u['cost_date'])].copy()

    if not has_scope:
        u['scope'] = 'all'
        u['bucket'] = np.where(u['population'] == ORGANIC_LABEL, 'organic', 'acquired')

    u = u.drop_duplicates(subset=['id'])

    r = revenue_df[['playerid', 'date', 'amount']].copy()
    r['date'] = pd.to_datetime(r['date'], errors='coerce').dt.date
    r = r.loc[pd.notna(r['date'])].copy()
    if positive_amount_only:
        r = r[r['amount'] > 0]

    rr = r.merge(
        u.rename(columns={'id': '__uid__'}),
        left_on='playerid', right_on='__uid__', how='inner'
    )
    rr['dsi'] = (pd.to_datetime(rr['date']) - pd.to_datetime(rr['cost_date'])).dt.days
    rr = rr.loc[(rr['dsi'] >= 0) & (rr['dsi'] <= (max(checkpoints) - 1))].copy()

    daily_user = (
        rr.groupby(['scope', 'bucket', 'cost_date', '__uid__', 'dsi'], observed=True)['amount']
          .sum().reset_index()
          .sort_values(['scope', 'bucket', 'cost_date', '__uid__', 'dsi'])
    )
    daily_user['cum_amount'] = (
        daily_user.groupby(['scope', 'bucket', 'cost_date', '__uid__'], observed=True)['amount']
                  .cumsum()
    )

    rows = []
    scopes = sorted(u['scope'].unique())

    for horizon in goal_horizons:
        cohort_end   = (as_of_date - pd.Timedelta(days=horizon)).date()
        cohort_start = (as_of_date - pd.Timedelta(days=horizon + (lookback_cohorts - 1))).date()

        elig = u.loc[
            (u['cost_date'] >= cohort_start) &
            (u['cost_date'] <= cohort_end)
        ].copy()

        if elig.empty:
            print(f'[horizon={horizon}] No eligible users — skipping.')
            continue

        eligible_cps = sorted(c for c in checkpoints if c <= horizon)

        for scope in scopes:
            scope_elig = elig.loc[elig['scope'] == scope]
            if scope_elig.empty:
                continue

            scope_du_h = daily_user.merge(
                scope_elig[['scope', 'bucket', 'id', 'cost_date']].rename(columns={'id': '__uid__'}),
                on=['scope', 'bucket', '__uid__', 'cost_date'], how='inner'
            )

            excluded = {
                cfg['label']: set()
                for cfg in trim_configs
                if cfg['method'] == 'cohort_trim' and persistent_trim
            }

            for cp in eligible_cps:
                du_cp = scope_du_h.loc[scope_du_h['dsi'] <= (cp - 1)]

                row = dict(
                    scope          = scope,
                    goal_horizon   = horizon,
                    cohort_start   = cohort_start,
                    cohort_end     = cohort_end,
                    checkpoint_day = cp,
                )

                if du_cp.empty:
                    for cfg in trim_configs:
                        lbl = cfg['label']
                        row.update({
                            f'organic_sum_{lbl}'       : 0.0,
                            f'acquired_sum_{lbl}'      : 0.0,
                            f'total_sum_{lbl}'         : 0.0,
                            f'organic_share_pct_{lbl}' : np.nan,
                            f'users_org_{lbl}'         : 0,
                            f'users_acq_{lbl}'         : 0,
                        })
                    rows.append(row)
                    continue

                cum_cp = (
                    du_cp.groupby(['bucket', 'cost_date', '__uid__'], observed=True)['cum_amount']
                         .max().reset_index(name='cum_cp')
                )
                cum_cp = (
                    scope_elig.rename(columns={'id': '__uid__'})
                              .merge(cum_cp, on=['bucket', 'cost_date', '__uid__'], how='left')
                )
                cum_cp['cum_cp'] = cum_cp['cum_cp'].fillna(0.0)

                print_line = f'  [{scope}] D{cp:>3}:'

                for cfg in trim_configs:
                    method, pct, lbl = cfg['method'], cfg['pct'], cfg['label']

                    if method == 'none' or pct == 0.0:
                        kept = cum_cp.copy()

                    elif method == 'cohort_trim':
                        working = cum_cp.copy()
                        if persistent_trim and excluded.get(lbl):
                            working = working.loc[~working['__uid__'].isin(excluded[lbl])]

                        depositors = working.loc[working['cum_cp'] > 0]
                        if not depositors.empty:
                            thresh_map = (
                                depositors.groupby('bucket', observed=True)['cum_cp']
                                          .quantile(1.0 - pct)
                                          .to_dict()
                            )
                            thresh_series = working['bucket'].map(thresh_map).fillna(np.inf)
                            above_mask = (working['cum_cp'] > 0) & (working['cum_cp'] > thresh_series)
                            if persistent_trim:
                                excluded[lbl] |= set(working.loc[above_mask, '__uid__'].unique())
                            kept = working.loc[~above_mask]
                        else:
                            kept = working

                    elif method == 'winsor':
                        depositors = cum_cp.loc[cum_cp['cum_cp'] > 0]
                        if not depositors.empty:
                            cap_map = (
                                depositors.groupby('bucket', observed=True)['cum_cp']
                                          .quantile(1.0 - pct)
                                          .to_dict()
                            )
                            kept = cum_cp.copy()
                            caps = kept['bucket'].map(cap_map).fillna(np.inf)
                            kept['cum_cp'] = np.minimum(kept['cum_cp'], caps)
                        else:
                            kept = cum_cp.copy()

                    else:
                        raise ValueError(f"Unknown trim method: {method}")

                    sums = kept.groupby('bucket', observed=True)['cum_cp'].sum().to_dict()
                    cnts = kept.groupby('bucket', observed=True)['__uid__'].nunique().to_dict()

                    org_sum = float(sums.get('organic',  0.0))
                    acq_sum = float(sums.get('acquired', 0.0))
                    total   = org_sum + acq_sum
                    share   = (org_sum / total) if total > 0 else np.nan

                    row.update({
                        f'organic_sum_{lbl}'       : org_sum,
                        f'acquired_sum_{lbl}'      : acq_sum,
                        f'total_sum_{lbl}'         : total,
                        f'organic_share_pct_{lbl}' : share,
                        f'users_org_{lbl}'         : int(cnts.get('organic',  0)),
                        f'users_acq_{lbl}'         : int(cnts.get('acquired', 0)),
                    })

                    print_line += (f'  [{lbl}] org={org_sum:>10,.0f} '
                                   f'acq={acq_sum:>10,.0f} share={share:.1%}')

                print(print_line)
                rows.append(row)

    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows).sort_values(['scope', 'goal_horizon', 'checkpoint_day']).reset_index(drop=True)
    id_cols = ['scope', 'goal_horizon', 'cohort_start', 'cohort_end', 'checkpoint_day']
    metric_cols = []
    for cfg in trim_configs:
        lbl = cfg['label']
        metric_cols += [
            f'organic_sum_{lbl}', f'acquired_sum_{lbl}', f'total_sum_{lbl}',
            f'organic_share_pct_{lbl}', f'users_org_{lbl}', f'users_acq_{lbl}',
        ]
    return df[id_cols + metric_cols]


print('Organic share function defined.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# PART 1 — Build per-population ARPU curves
# ══════════════════════════════════════════════════════════════

print(f'as_of_date = {AS_OF_DATE.date()}')
print('Building base tables for per-population ARPU curves...')

u_pop, daily_pop = build_user_revenue_cums(
    users_df.loc[users_df['population'].isin(POPULATIONS)].copy(),
    revenue_df,
    max_day=365,
)

cv_pop_df, curve_pop_df = build_all_populations(
    u_pop, daily_pop, as_of_date=AS_OF_DATE, debug=True
)

# ══════════════════════════════════════════════════════════════
# PART 2 — Build BLENDED ARPU curve
# Pool every user (Web/App/Affiliate/PPC/Organic) into one bucket.
# ══════════════════════════════════════════════════════════════

print('\n' + '=' * 60)
print('Building base tables for BLENDED population...')
print('=' * 60)

users_blend = users_df.copy()
users_blend['population'] = 'Blended'

u_blend, daily_blend = build_user_revenue_cums(
    users_blend, revenue_df, max_day=365
)

# Run the same pipeline against the synthetic 'Blended' population.
_BLENDED_POPS_BACKUP = POPULATIONS
POPULATIONS = ['Blended']
cv_blend_df, curve_blend_df = build_all_populations(
    u_blend, daily_blend, as_of_date=AS_OF_DATE, debug=True
)
POPULATIONS = _BLENDED_POPS_BACKUP

# Concatenate per-pop and blended results
cv_df    = pd.concat([cv_pop_df,    cv_blend_df],    ignore_index=True)
curve_df = pd.concat([curve_pop_df, curve_blend_df], ignore_index=True)

print('\n' + '=' * 75)
print('CV SUMMARY — ALL POPULATIONS + BLENDED')
print('=' * 75)
display_cols = [
    'population', 'patch', 'cohort_start', 'cohort_end',
    'cv_before', 'cv_after',
    'n_cohort_dates_total', 'n_cohort_dates_kept',
    'n_users_excluded_prior', 'n_users_pre_trim', 'n_users_post_trim',
    'total_rev_before_trim', 'total_rev_after_trim',
    'flagged', 'trim_method',
]
print(cv_df[display_cols].to_string(index=False))

flagged_patches = cv_df.loc[cv_df['flagged'] == True]
if not flagged_patches.empty:
    print('\n>>> FLAGGED PATCHES (kept as-is, no fallback):')
    for _, r in flagged_patches.iterrows():
        print(f"    {r['population']} / {r['patch']}  CV={r['cv_after']:.4f}")
else:
    print('\nAll patches clean — no flags.')

milestones = [1, 7, 14, 30, 60, 90, 120, 150, 180, 210, 240, 270, 365]
for pop in POPULATIONS + ['Blended']:
    sub = curve_df.loc[curve_df['population'] == pop]
    if sub.empty:
        continue
    print(f'\n{"=" * 45}')
    print(f'ARPU CURVE — {pop}')
    print(f'{"=" * 45}')
    ms = sub.loc[sub['day'].isin(milestones)][['day', 'ARPU_nominal', 'effective_patch']]
    print(ms.to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════
# PART 3 — Organic share (single config, per scope × goal horizon)
# Uses scope+bucket from users_df (app vs non_app for RP).
# Applied per-population in goal construction via scope mapping.
# ══════════════════════════════════════════════════════════════

org_label = f'{ORGANIC_TRIM_METHOD}_{round(ORGANIC_TRIM_PCT * 100):g}pct'
org_config = [{'method': ORGANIC_TRIM_METHOD, 'pct': ORGANIC_TRIM_PCT, 'label': org_label}]
print(f'Computing organic share [{org_label}] for all goal horizons...')

organic_full = organic_share_cohort_progression(
    users_df, revenue_df,
    as_of_date=AS_OF_DATE,
    goal_horizons=GOAL_HORIZONS,
    checkpoints=CHECKPOINTS,
    lookback_cohorts=LOOKBACK_COHORTS,
    positive_amount_only=True,
    trim_configs=org_config,
    persistent_trim=True,
)

organic_df = organic_full[['scope', 'goal_horizon', 'cohort_start', 'cohort_end',
                           'checkpoint_day',
                           f'organic_share_pct_{org_label}',
                           f'users_org_{org_label}',
                           f'users_acq_{org_label}']].copy()
organic_df = organic_df.rename(columns={
    f'organic_share_pct_{org_label}': 'organic_share_pct',
    f'users_org_{org_label}': 'users_org',
    f'users_acq_{org_label}': 'users_acq',
})

print(f'\nOrganic share rows: {len(organic_df)}')
print(f'Trim config: {ORGANIC_TRIM_METHOD} {ORGANIC_TRIM_PCT:.0%}')
for scope in organic_df['scope'].unique():
    print(f'\n── Organic share: scope={scope} ──')
    print(
        organic_df.loc[organic_df['scope'] == scope,
                       ['scope', 'goal_horizon', 'checkpoint_day',
                        'organic_share_pct', 'users_org', 'users_acq']]
        .to_string(index=False)
    )

In [ ]:
# ══════════════════════════════════════════════════════════════
# PART 4 — Goal construction
# Per-population: adjusted = (ARPU_day / ARPU_horizon) × (1 − organic_share_at_horizon)
# Blended:        adjusted = raw_goal_ratio (no organic adjustment)
# ──
# Organic share rule: for every day inside a horizon, use the share measured AT
# the horizon endpoint (not the share at the day's nearest checkpoint). Decided
# by the marketing team — keeps the share constant within a horizon.
# ══════════════════════════════════════════════════════════════

def build_goals(curve_df, organic_df, populations, goal_horizons=GOAL_HORIZONS,
                organic_share_cap_horizon=ORGANIC_SHARE_CAP_HORIZON):
    has_scope = 'scope' in organic_df.columns
    available_scopes = set(organic_df['scope'].unique()) if has_scope else {'all'}

    # Lookup keyed by (scope, horizon) → organic_share_pct
    org_lookup = {}
    scope_col = 'scope' if has_scope else None
    endpoint = organic_df.loc[organic_df['checkpoint_day'] == organic_df['goal_horizon']]
    if scope_col:
        for scope, grp in endpoint.groupby(scope_col):
            org_lookup[scope] = grp.set_index('goal_horizon')['organic_share_pct'].to_dict()
    else:
        org_lookup['all'] = endpoint.set_index('goal_horizon')['organic_share_pct'].to_dict()

    def pop_to_scope(pop):
        if pop == 'App' and 'app' in available_scopes:
            return 'app'
        if 'non_app' in available_scopes:
            return 'non_app'
        return next(iter(available_scopes))

    rows = []
    all_pops = list(populations) + ['Blended']

    for pop in all_pops:
        is_blended = (pop == 'Blended')

        pop_curve = curve_df.loc[curve_df['population'] == pop].copy()
        if pop_curve.empty:
            continue
        pop_curve = pop_curve.drop_duplicates(subset='day').set_index('day')

        for horizon in goal_horizons:
            if horizon not in pop_curve.index:
                print(f'[{pop}] Day {horizon} missing from curve — skipping horizon.')
                continue
            arpu_horizon = pop_curve.loc[horizon, 'ARPU_nominal']
            if not np.isfinite(arpu_horizon) or arpu_horizon == 0:
                continue

            if is_blended:
                org_share = 0.0
            else:
                scope = pop_to_scope(pop)
                lookup_horizon = min(horizon, organic_share_cap_horizon)
                org_share = org_lookup.get(scope, {}).get(lookup_horizon, np.nan)

            for day in range(1, horizon + 1):
                if day not in pop_curve.index:
                    continue
                arpu_day  = pop_curve.loc[day, 'ARPU_nominal']
                eff_patch = pop_curve.loc[day, 'effective_patch'] if 'effective_patch' in pop_curve.columns else ''
                is_extrap = bool(pop_curve.loc[day, 'is_extrapolated']) if 'is_extrapolated' in pop_curve.columns else False
                raw_goal  = arpu_day / arpu_horizon

                if np.isfinite(org_share):
                    adj_goal = raw_goal * (1 - org_share)
                else:
                    adj_goal = np.nan

                rows.append(dict(
                    population           = pop,
                    goal_horizon         = horizon,
                    day                  = day,
                    ARPU_nominal         = float(arpu_day),
                    ARPU_at_horizon      = float(arpu_horizon),
                    raw_goal_ratio       = float(raw_goal),
                    organic_share        = float(org_share) if np.isfinite(org_share) else None,
                    adjusted_goal_ratio  = float(adj_goal)  if np.isfinite(adj_goal)  else None,
                    effective_patch      = eff_patch,
                    is_extrapolated      = is_extrap,
                ))

    return pd.DataFrame(rows)


goals_df = build_goals(curve_df, organic_df, POPULATIONS)
print(f'Goals rows: {len(goals_df):,}')

preview_days = [1, 7, 14, 30, 60, 90, 120, 150, 180, 210, 240, 270, 365]
preview_cols = ['day', 'raw_goal_ratio',
                'organic_share', 'adjusted_goal_ratio']

for pop in list(POPULATIONS) + ['Blended']:
    for horizon in GOAL_HORIZONS:
        sub = goals_df.loc[
            (goals_df['population'] == pop) &
            (goals_df['goal_horizon'] == horizon) &
            (goals_df['day'].isin(preview_days))
        ]
        if sub.empty:
            continue
        print(f'\n── {pop} │ Goal horizon D{horizon} ──')
        print(sub[preview_cols].to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════
# EXPORT
# ══════════════════════════════════════════════════════════════

cv_df.to_csv('realprize_combined_cv_summary.csv', index=False)
curve_df.to_csv('realprize_combined_arpu_curve.csv', index=False)
organic_df.to_csv('realprize_combined_organic_share.csv', index=False)
goals_df.to_csv('realprize_combined_goals_adjusted.csv', index=False)

print('Saved:')
print('  realprize_combined_cv_summary.csv')
print('  realprize_combined_arpu_curve.csv')
print('  realprize_combined_organic_share.csv')
print('  realprize_combined_goals_adjusted.csv')

from google.colab import files
files.download('realprize_combined_cv_summary.csv')
files.download('realprize_combined_arpu_curve.csv')
files.download('realprize_combined_organic_share.csv')
files.download('realprize_combined_goals_adjusted.csv')